# Vergleich von Studienkredit-Tilgungsplänen mit PROC LOAN

## Zusammenfassung

Eine Studienfinanzierungsstelle prüft, wie eine Absolventenkohorte einen repräsentativen Kreditsaldo von 27.500 $ zurückzahlen sollte, indem vier Tilgungsstrukturen verglichen werden — ein staatlicher Festzins-Standardplan, eine private Umschuldung mit Bearbeitungsgebühr, ein gedeckelter variabel verzinslicher Kredit (ARM) und ein arbeitgebergefördertes Buydown-Modell — mithilfe von `PROC LOAN`. Über eine Laufzeit von 120 Monaten liegen die vier Angebote bei monatlicher Rate und Gesamtzinsen zu ihren genannten Startzinssätzen klar auseinander: Der staatliche Standardplan kostet die meisten Zinsen (**10.021,22 $** bei 6,53 %, Rate **312,68 $**), die private Umschuldung senkt den Zinssatz, erhebt aber eine Gebühr von 412,50 $ (**9.120,20 $** bei 5,99 %, Rate **305,17 $**), und der günstiger notierte ARM (**7.099,76 $** bei 4,75 %, Rate **288,33 $**) sowie das Arbeitgeber-Buydown (**6.700,67 $** bei 4,50 %, Rate **285,01 $**) verursachen die niedrigsten Zinskosten. Ein `COMPARE`-Block berichtet anschließend für jeden Plan die kumulierten Zinsen, die Tilgung und den ausstehenden Saldo nach 36, 60 und 120 Monaten und zeigt, wie weit jeder Kredit an den Punkten getilgt ist, an denen ein Kreditnehmer am ehesten umschulden oder vorzeitig zurückzahlen würde.

## Datenquellen

| Datensatz | Zeilen | Beschreibung | Schlüsselvariablen |
|---------|------|-------------|---------------|
| `borrowers` | 40 | Synthetische Kreditprofile einer Absolventenkohorte, inline erzeugt mit `call streaminit(20260531)` und `rand('uniform')`. Dienen dazu, realistische Kreditbedingungen für den Vergleich zu motivieren. | `student_id` (1001-1040), `balance` (Kapital bei Abschluss; diese Ziehung reicht von 11.800 $ bis 47.750 $, Mittelwert 30.771 $), `apr` (4,10 %-9,10 % nomineller Jahreszins, Mittelwert 6,50 %), `term` (120 oder 180 Monate, Mittelwert 144), `origination_pct` (1,0 %-2,0 % Gebühr, Mittelwert 1,50 %) |

Der repräsentative Kreditnehmer, der mit `PROC LOAN` analysiert wird (Betrag 27.500 $, 120-monatige Laufzeit, Start Juli 2026), liegt im unteren Mittelfeld der Saldo- und Zinsverteilung dieser Kohorte; es werden keine externen Daten oder Netzwerkdaten verwendet. Die Kohorte dient dazu, plausible Kreditbedingungen zu motivieren — der direkte Vergleich erfolgt am einzelnen repräsentativen Kredit.

# Vergleich von Studienkredit-Tilgungsplänen mit PROC LOAN

Wenn Studierende ihren Abschluss machen, muss eine Studienfinanzierungsstelle ihnen helfen, unter konkurrierenden Tilgungsangeboten zu wählen. `PROC LOAN` (SAS/ETS) ist genau dafür gebaut: Es tilgt Festzins-, variabel verzinsliche (ARM-) und Buydown-Kredite und vergleicht sie anschließend nebeneinander nach Rate, Gesamtzinsen und Tilgungsfortschritt.

In diesem Notebook werden wir:

1. Eine synthetische Absolventenkohorte erzeugen, um realistische Kreditbedingungen festzulegen.
2. Die Kohorte mit `PROC MEANS` zusammenfassen.
3. Vier Tilgungspläne für einen repräsentativen Saldo von 27.500 $ aufstellen und mit `PROC LOAN` vergleichen (FIXED / ARM / BUYDOWN + COMPARE).
4. Den empfohlenen Festzinsplan eigenständig erneut ausführen, um seine eigenständige Wirtschaftlichkeit zu bestätigen.

Alles läuft lokal auf Inline-Synthetikdaten — kein Zugriff auf externe Dateien oder Netzwerke.

## Schritt 1 — Eine synthetische Absolventenkohorte erzeugen

Wir simulieren 40 Kreditnehmer. Jeder zieht einen Kapitalsaldo bei Abschluss, einen effektiven Jahreszins, der lose an die Bonität gekoppelt ist, eine Standard-Tilgungslaufzeit (10 oder 15 Jahre) und einen Bearbeitungsgebühr-Prozentsatz. Der Seed macht die Daten reproduzierbar.

In [1]:
DATEN borrowers;
   AUFRUFEN streaminit(20260531);
   LÄNGE plan $ 28;
   AUSFÜHRUNG student_id = 1001 BIS 1040;
      /* Kapitalsaldo bei Abschluss: 9.500 $ - 48.000 $ */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* Nomineller Jahreszins nach Bonitätsstufe: 4,0 % - 9,5 % */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* Standard-Tilgungslaufzeit: 120 oder 180 Monate */
      WENN rand('uniform') < 0.6 DANN term = 120;
      SONST term = 180;
      /* Bearbeitungsgebühr als Prozentsatz des Kapitals: 1,0 % - 2,0 % */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      AUSGABE;
   ENDE;
AUSFÜHREN;


NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Schritt 2 — Die Kohorte profilieren

Bevor wir einzelne Kredite modellieren, betrachten wir die Verteilung von Salden, Zinssätzen und Laufzeiten. Das zeigt uns, wie ein *repräsentativer* Kreditnehmer aussieht — die Grundlage für den direkten Vergleich, der folgt.

In [2]:
PROZEDUR MITTELWERTE DATEN=borrowers n mean MIN MAX maxdec=2;
   VAR balance apr term origination_pct;
   BEZEICHNUNG balance="Kreditsaldo ($)" apr="Effektivzins (%)" term="Laufzeit (Monate)" origination_pct="Bearbeitungsgebühr (%)";
AUSFÜHREN;

                                                  The MEANS Procedure

 Variable         Label                          N           Mean     Minimum     Maximum
 ----------------------------------------------------------------------------------------
 balance          Kreditsaldo ($)               40       30771.25    11800.00    47750.00
 apr              Effektivzins (%)              40           6.50        4.10        9.10
 term             Laufzeit (Monate)             40         144.00      120.00      180.00
 origination_pct  Bearbeitungsgebühr (%)        40           1.50        1.00        2.00
 ----------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Schritt 3 — Vier Tilgungspläne für einen repräsentativen Kreditnehmer vergleichen

Anhand eines repräsentativen Saldos von 27.500 $ über eine 120-monatige (10-jährige) Laufzeit mit Start im Juli 2026 stellen wir vier realistische Angebote nebeneinander:

- **Staatlicher Studienkredit (Standard, ohne Subvention)** — ein einfacher Festzinskredit zu 6,53 %.
- **Private Umschuldung (mit Gebühr)** — ein niedrigerer Festzins von 5,99 %, aber mit Einrichtungskosten von 412,50 $ (`INIT=`).
- **Private variable Rate (ARM)** — ein variabler Kredit zu 4,75 % mit einer Deckelung von 1 % pro Anpassung / 5 % über die Laufzeit (`CAPS=`), einem `MAXRATE=` von 9,75 %, jährlichem `ADJUSTFREQ=` und dem Schlüsselwort `WORSTCASE`.
- **Arbeitgeber-Buydown 2-1** — ein subventionierter Start bei 4,50 %, der laut Zeitplan über `BDRATES=` im 2. Jahr auf 5,50 % und danach auf 6,50 % steigt.

Die `COMPARE`-Anweisung fordert die kreditübergreifende Sicht nach 36, 60 und 120 Monaten an, mit einem `TAXRATE=` von 22 % und einer minimal attraktiven Verzinsung (`MARR=`) von 4 %; `OUTSUM=` und `OUTCOMP=` erfassen die Zusammenfassung je Kredit und die Vergleichszeilen. Die untenstehende Liste berichtet für jeden Plan und jeden Prüfzeitpunkt die **kumulierten gezahlten Zinsen, die kumulierte getilgte Tilgung und den ausstehenden Saldo** — eine klare Sicht auf den Tilgungsfortschritt über alle Kandidaten hinweg.

> **Hinweis zu Zinsanpassungen.** Jenners `PROC LOAN` tilgt jeden Plan zu seinem genannten **Start**zinssatz, sodass die `CAPS`/`MAXRATE`/`WORSTCASE`-Einstellungen des ARM und die `BDRATES`-Stufen des Buydowns in der Liste zwar als Kreditbedingungen ausgegeben werden, aber **nicht** in die Zahlungsberechnung einfließen — die unten stehenden ARM- und Buydown-Zahlen spiegeln ihre Startzinssätze von 4,75 % bzw. 4,50 % wider, konstant über die gesamte Laufzeit gehalten. Betrachten Sie diese beiden Summen als Best-Case-Kosten (zum Startzinssatz), nicht als Worst-Case-Obergrenzen.

In [3]:
PROZEDUR loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         BEZEICHNUNG="Staatlicher Studienkredit (Standard)";

   fixed rate=5.99 init=412.50
         BEZEICHNUNG="Private Umschuldung (mit Gebühr)";

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       BEZEICHNUNG="Private variable Rate (ARM, ungünstigster Fall)";

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           BEZEICHNUNG="Arbeitgeber-Buydown 2-1, danach 6,5%";

   COMPARE at=(36 60 120) ALL taxrate=22 marr=4 OUTCOMP=plan_cmp;
AUSFÜHREN;

                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: Staatlicher Studienkredit (Standard)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: Private Umschuldung (mit Gebühr)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: Private variable Rate (ARM, ungünstigster Fall)
    Loan Type:                   Adjustable Rate
    Amount (P


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Schritt 4 — Den empfohlenen Festzinsplan eigenständig erneut ausführen

Für den Kreditnehmer, dem Zahlungssicherheit wichtig ist, ist der staatliche Festzins-Standardplan die sichere Standardwahl. Wir führen ihn separat erneut aus, um seine wichtigsten Kennzahlen zu bestätigen: dieselbe monatliche Rate von **312,68 $**, insgesamt gezahlte **37.521,22 $** und Gesamtzinsen von **10.021,22 $**, die bereits im Vier-Wege-Vergleich zu sehen waren, nun als eigenständige Kreditübersicht dargestellt.

In [4]:
PROZEDUR loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed BEZEICHNUNG="Staatlicher Studienkredit (Standard)";
AUSFÜHREN;

                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: Staatlicher Studienkredit (Standard)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: Staatlicher Studienkredit (Standard)
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47        2151.66         23332.34
         3         23332.34        3752.12        1455.68        2296.44         21035.90
         4         21035.90


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Ergebnisse interpretieren

Alle vier Pläne tilgen dasselbe Kapital von 27.500 $ vollständig über 120 Monate (der ausstehende Saldo jedes Plans erreicht in der COMPARE-Tabelle bei Monat 120 den Wert 0,00 $), sodass die Entscheidung von der **monatlichen Rate** und den **Gesamtzinsen zum genannten Zinssatz** abhängt:

| Plan | Zinssatz | Rate | Gesamtzinsen | Anmerkungen |
|------|---------|------|----------------|-------|
| Staatlicher Standardkredit | 6,53 % | 312,68 $ | 10.021,22 $ | Keine Gebühr; stärkster Kreditnehmerschutz |
| Private Umschuldung | 5,99 % | 305,17 $ | 9.120,20 $ | Bearbeitungsgebühr von 412,50 $ |
| Private variable Rate (ARM) | 4,75 % (Start) | 288,33 $ | 7.099,76 $ | Zinssatz kann steigen; Betrag sind die Kosten zum Startzinssatz |
| Arbeitgeber-Buydown 2-1 | 4,50 % (Start) | 285,01 $ | 6.700,67 $ | Abhängig von fortlaufender Beschäftigung |

- Der **staatliche Standardplan** ist bei den Zinsen am teuersten (10.021,22 $), bietet aber eine feste, vorhersehbare Rate von 312,68 $ ohne Gebühr.
- Die **private Umschuldung** senkt den Zinssatz auf 5,99 % (9.120,20 $ Zinsen, 901 $ weniger als beim staatlichen Plan), erhebt aber eine Bearbeitungsgebühr von 412,50 $, sodass ihr Nettovorteil gegenüber dem staatlichen Plan bei etwa 488 $ Zinsersparnis abzüglich Gebühr liegt — nur relevant, wenn der Kredit lange genug läuft, um nicht vorher umgeschuldet zu werden.
- Der **ARM** und der **Buydown** zeigen hier die niedrigsten Zinsen (7.099,76 $ und 6.700,67 $) rein deshalb, weil ihre **Startzinssätze** deutlich unter den Festzinsangeboten liegen. Wie in Schritt 3 angemerkt, hält Jenner diese Startzinssätze konstant, sodass dies Best-Case-Zahlen sind: Ein realer ARM, der nach oben angepasst wird — oder ein Buydown, der seine Arbeitgeber-Subvention verliert — würde höher ausfallen, und die Rate ist nicht garantiert.

Die `COMPARE`-Tabelle verdeutlicht dies, indem sie zeigt, wie schnell jeder Plan tilgt. Nach 36 Monaten hat der staatliche Plan 4.792,27 $ Zinsen gezahlt und 6.464,10 $ Tilgung geleistet (Saldo 21.035,90 $), während der Buydown nur 3.263,97 $ Zinsen gezahlt, aber 6.996,24 $ Tilgung geleistet hat (Saldo 20.503,76 $) — die Pläne mit niedrigerem Zinssatz kosten in den ersten drei Jahren sowohl weniger Zinsen als auch tilgen schneller. Für einen risikoscheuen Absolventen rechtfertigt die Zinssicherheit des staatlichen Standardplans oft die höheren Zinsen; für einen Kreditnehmer, der sich einer stabilen, dauerhaften Beschäftigung sicher ist, liefert der subventionierte Start des Buydowns die größten Einsparungen unter den Optionen, die ihren Zinssatz tatsächlich festschreiben.